# Week 2, Day 5: Random numbers and simulation: Poisson spike trains

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w002/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)
print(rng.random(3))            # uniform on [0, 1)
print(rng.normal(0, 1, 3))      # Gaussian with mean 0, SD 1
print(rng.poisson(4.0, 5))      # Poisson counts with mean 4
print(rng.exponential(0.05, 3)) # exponential with mean 0.05

rng2 = np.random.default_rng(seed=42)
print("same seed, same numbers:", rng2.random(3))

In [ ]:
def poisson_train(rate, T, rng, refractory=0.0):
    """Spike times in [0, T) from exponential intervals, plus an optional dead time after each spike."""
    times, t = [], 0.0
    while True:
        t += refractory + rng.exponential(1 / rate)
        if t >= T:
            return np.array(times)
        times.append(t)

rng = np.random.default_rng(0)
for ref in [0.0, 0.002, 0.01]:
    isi = np.diff(poisson_train(50, 200, rng, refractory=ref))
    print(f"refractory {ref*1000:4.0f} ms: mean ISI {isi.mean()*1000:5.1f} ms, CV {isi.std()/isi.mean():.2f}")

### Exercise: Poisson spikes from bins

Simulate 500 trials of 1 s at 20 Hz using 1 ms bins (method 1). Store a boolean array `spikes` of shape (500, 1000) where True means a spike in that bin.

In [ ]:
rng = np.random.default_rng(1)
rate, dt = 20.0, 0.001
spikes = ...
print(spikes.shape, "mean rate:", spikes.sum(axis=1).mean(), "Hz")

In [ ]:
import numpy as np
def test_shape():
    "spikes is a boolean array of shape (trials, bins)"
    assert spikes.shape == (500, 1000) and spikes.dtype == bool

def test_rate():
    "the mean rate is close to 20 Hz"
    r = spikes.sum(axis=1).mean() / 1.0
    assert 18.5 < r < 21.5, f"Mean rate was {r:.1f} Hz"

test_shape()
test_rate()
print("All checks passed")

### Exercise: Fano factor

Write `fano(counts)` (variance with $n-1$ divided by mean). Compute `F` for the spike counts per trial from `spikes`.

In [ ]:
def fano(counts):
    ...

F = ...
print("Fano factor:", F)

In [ ]:
import numpy as np
def test_fano_function():
    "fano computes variance over mean of counts"
    assert abs(fano(np.array([2, 4, 4, 6])) - np.var([2, 4, 4, 6], ddof=1) / 4) < 1e-12

def test_fano_value():
    "the Fano factor of the simulated counts is close to 1"
    assert 0.85 < F < 1.15, f"F was {F:.2f}"

test_fano_function()
test_fano_value()
print("All checks passed")

### Exercise: Refractoriness and regularity

For a 40 Hz neuron, compute the ISI CV for refractory periods `refs = [0, 0.002, 0.005, 0.01, 0.02]` seconds using `poisson_train` with T = 200 s. Store them in `cvs` and plot CV against refractory period.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)
refs = [0, 0.002, 0.005, 0.01, 0.02]
cvs = []
for ref in refs:
    ...
print(np.round(cvs, 3))
plt.figure(figsize=(4, 3)); plt.plot(np.array(refs) * 1000, cvs, "o-")
plt.xlabel("refractory period (ms)"); plt.ylabel("ISI CV"); plt.show()

In [ ]:
import numpy as np
def test_lengths():
    "one CV per refractory period"
    assert len(cvs) == len(refs)

def test_decreasing():
    "CV falls as the refractory period grows"
    assert all(b < a for a, b in zip(cvs, cvs[1:])), f"cvs = {np.round(cvs, 3)}"
    assert abs(cvs[0] - 1) < 0.08, "With no refractory period the CV should be close to 1."

test_lengths()
test_decreasing()
print("All checks passed")